In [ ]:
# Velocity, pseudotime and CellRank analysis.
import scanpy as sc
import numpy as np
import pandas as pd  
import matplotlib.pyplot as plt
import matplotlib
import anndata as ad
import scvelo as scv
import palantir as pl
import cellrank as cr

In [ ]:
# Load data
adata = sc.read('/home/yanjun2/scanpy/8_figures/inte_palantir.h5ad')

In [ ]:
sc.settings.figdir = "/home/yanjun2/scanpy/8_figures"
sc.set_figure_params(scanpy=True, dpi=300, dpi_save=300, 
                         frameon=True, vector_friendly=True, fontsize=10, figsize= (5,5), color_map=None, format='pdf', 
                         facecolor=None, transparent=False, ipython_format='png2x')

In [ ]:
ldata1 = scv.read('/home/yc/Desktop/loom/R1_D1MDI_D1A.loom', cache=True)
ldata2 = scv.read('/home/yc/Desktop/loom/R1_D2MDI_D2A.loom', cache=True)
ldata3 = scv.read('/home/yc/Desktop/loom/R1_D3MDI_D3A.loom', cache=True)
ldata4 = scv.read('/home/yc/Desktop/loom/R1_D4MDI_D4A.loom', cache=True)
ldata5 = scv.read('/home/yc/Desktop/loom/R1_D5MDI_D5A.loom', cache=True)
#ldata6 = scv.read('/home/yc/Desktop/loom/R1_DMo_D0M.loom', cache=True)

In [ ]:
def clean_cell_ids(ldata):
    ldata.obs.index = ldata.obs.index.str.replace(r'^.*:', '', regex=True)
    ldata.obs.index = ldata.obs.index.str.replace(r'x$', '', regex=True)
    return ldata
ldata1 = clean_cell_ids(ldata1)
ldata2 = clean_cell_ids(ldata2)
ldata3 = clean_cell_ids(ldata3)
ldata4 = clean_cell_ids(ldata4)
ldata5 = clean_cell_ids(ldata5)

In [ ]:
ldata1.var_names_make_unique()
ldata2.var_names_make_unique()
ldata3.var_names_make_unique()
ldata4.var_names_make_unique()
ldata5.var_names_make_unique()

In [ ]:
ldata = ldata1.concatenate([ldata2, ldata3, ldata4, ldata5
                            ])

In [ ]:
adata.obs_names = adata.obs_names.str.split('-').str[0]

In [ ]:
adata = scv.utils.merge(adata, ldata)

In [ ]:
# PAGA and diffusion maps
sc.tl.paga(adata, 'leiden_res_0_8')
scv.pl.paga(adata, basis='umap', threshold=0.4, max_edge_width=0.3, alpha=.4, title='',  
            figsize=(4.7,5), legend_loc='right margin')

In [ ]:
adata.X = adata.layers['spliced']
adata.X = adata.layers['unspliced']

In [ ]:
# Neighbours, UMAP and clustering
scv.pp.filter_and_normalize(adata)
sc.pp.neighbors(adata, n_pcs=30, n_neighbors=30, random_state=0)
scv.pp.moments(adata, n_pcs=None, n_neighbors=None)

In [ ]:
scv.tl.velocity(adata, mode = 'stochastic')
scv.tl.velocity_graph(adata)

In [ ]:
adata.obsm['exp_diffmap'] = adata.obsm['X_diffmap']

In [ ]:
scv.pl.velocity_embedding_stream(adata, basis='umap', color = 'leiden_res_0_8', legend_loc='right margin', arrow_size = 0.5, 
                                 size = 100#, save='velo_mdixwithouit.pdf'
                                )

In [ ]:
scv.tl.velocity_pseudotime(adata)
scv.pl.scatter(adata, color='velocity_pseudotime', cmap='RdBu_r')

In [ ]:
sc.pl.umap(adata, color='velocity_pseudotime', color_map='RdBu_r', size = 50)

In [ ]:
sc.pl.umap(adata, color='palantir_pseudotime', color_map='RdBu_r', size = 50)

In [ ]:
sc.pl.scatter(adata, x = 'palantir_pseudotime', y = 'velocity_pseudotime', color = 'leiden_res_0_8', size = 20)

In [ ]:
scv.pl.velocity_embedding_stream(adata, basis='diffmap', color = 'leiden_res_0_8', legend_loc='right margin', arrow_size = 0.5, 
                                 size = 100#, save='velo_mdixwithouit.pdf'
                                )

In [ ]:
scv.pl.velocity_embedding_stream(adata, basis='diffmap', color = 'velocity_pseudotime', legend_loc='right margin', arrow_size = 0.5, 
                                 size = 100, cmap = 'RdBu_r'
                                )

In [ ]:
scv.pl.velocity_embedding_stream(adata, basis='diffmap', color = 'palantir_pseudotime', legend_loc='right margin', arrow_size = 0.5, 
                                 size = 100, cmap = 'RdBu_r'
                                )

In [ ]:
sc.pl.violin(adata, 
             keys='velocity_pseudotime', 
             groupby='leiden_res_0_8', 
             order=['Mac1', 'Mac2', 'Mac3', 'Mac4', 'Mac5', 'Mac6', 'Mac7'],
             save='pt_vio.pdf')

In [ ]:
##cellrank##

In [ ]:
scv.settings.verbosity = 3
scv.settings.set_figure_params("scvelo")
cr.settings.verbosity = 2

In [ ]:
scv.pl.proportions(adata)

In [ ]:
adata.X = adata.layers['spliced']
adata.X = adata.layers['unspliced']

In [ ]:
scv.pp.filter_and_normalize(
    adata, min_shared_counts=20, n_top_genes=2000, subset_highly_variable=False
)

sc.tl.pca(adata)
sc.pp.neighbors(adata, n_pcs=30, n_neighbors=30, random_state=0)
scv.pp.moments(adata, n_pcs=None, n_neighbors=None)

In [ ]:
scv.tl.recover_dynamics(adata, n_jobs=12)

In [ ]:
scv.tl.velocity(adata, mode="dynamical")

In [ ]:
vk = cr.kernels.VelocityKernel(adata)

In [ ]:
vk.compute_transition_matrix()

In [ ]:
vk.plot_projection()

In [ ]:
adata

In [ ]:
scv.tl.velocity_graph(adata)
scv.pl.velocity_embedding_stream(adata, basis='umap', color = 'leiden_res_0_8', legend_loc='right margin', arrow_size = 0.5, 
                                 size = 100#, save='velo_mdixwithouit.pdf'
                                )

In [ ]:
vk.plot_random_walks(start_ixs={"leiden_res_0_8": "Mac1"}, max_iter=200, seed=0)

In [ ]:
vk.plot_random_walks(start_ixs={"leiden_res_0_8": "Mac2"}, max_iter=200, seed=0)

In [ ]:
timepoint_mapping = {'day1': 1, 'day2': 2, 'day3': 3, 'day4': 4, 'day5': 5}
adata.obs['timepoint_num'] = adata.obs['timepoint'].map(timepoint_mapping)
print(adata.obs['timepoint_num'].unique())


In [ ]:
adata.obs["timepoint_num"] = adata.obs["timepoint_num"].astype(float)

In [ ]:
sc.pl.embedding(
    adata,
    basis="umap",
    color=["timepoint_num", "leiden_res_0_8"],
    color_map="gnuplot",
)

In [ ]:
adata

In [ ]:
adata.X = adata.layers['data']

In [ ]:
sc.pp.pca(adata)
sc.pp.neighbors(adata, random_state=0)

In [ ]:
from cellrank.kernels import RealTimeKernel
from moscot.problems.time import TemporalProblem

In [ ]:
tp = TemporalProblem(adata)

In [ ]:
# Gene scoring
tp = tp.score_genes_for_marginals(
    gene_set_proliferation="human", gene_set_apoptosis="human"
)

In [ ]:
sc.pl.embedding(
    adata, basis="umap", color=["leiden_res_0_8", "proliferation", "apoptosis"], size = 100
)

In [ ]:
tp = tp.prepare(time_key="timepoint_num")

In [ ]:
tp = tp.solve(epsilon=1e-3, tau_a=0.95, scale_cost="mean")

In [ ]:
tmk = RealTimeKernel.from_moscot(tp)

In [ ]:
tmk.compute_transition_matrix(self_transitions="all", conn_weight=0.2, threshold="auto")

In [ ]:
tmk.plot_random_walks(
    max_iter=500,
    start_ixs={"timepoint_num": 1},
    basis="umap",
    seed=0,
    dpi=150,
    size=30,
)

In [ ]:
tmk.plot_random_walks(
    max_iter=500,
    start_ixs={"timepoint_num": 2},
    basis="umap",
    seed=0,
    dpi=150,
    size=30,
)

In [ ]:
ax = tmk.plot_single_flow(
    cluster_key="leiden_res_0_8",
    time_key="day",
    cluster="ME",
    min_flow=0.1,
    xticks_step_size=4,
    show=False,
    clusters=["MEF/other", "MET", "Stromal"],
)

_ = ax.set_xticklabels(ax.get_xticklabels(), rotation=90)

In [ ]:
combined_kernel = 0.8 * vk + 0.2 * tmk
combined_kernel

In [ ]:
from cellrank.estimators import GPCCA

g = GPCCA(combined_kernel)
print(g)

In [ ]:
g.fit(n_states=10, cluster_key="leiden_res_0_8")
g.plot_macrostates(which="all")

In [ ]:
g.predict_terminal_states(method="top_n", n_states=6)
g.plot_macrostates(which="terminal")

In [ ]:
g.compute_fate_probabilities()
g.plot_fate_probabilities(legend_loc="right")

In [ ]:
cr.pl.circular_projection(adata, keys="leiden_res_0_8", legend_loc="right")